In [2]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Este notebook se trabajo utilizando Kaggle en una configuración multi gpu, para probar que si servía puse
# algunos prints como verificación que si funcionaban ambas gpus T4
strategy = tf.distribute.MirroredStrategy()
print(f"GPUs disponibles: {strategy.num_replicas_in_sync}")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
GPUs disponibles: 2


In [3]:
DATA_DIR = '/kaggle/input/datasets/aryashah2k/mango-leaf-disease-dataset'
BATCH_SIZE = 32 * strategy.num_replicas_in_sync
IMG_SIZE = (224, 224)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
])

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(
    lambda x, y: (tf.keras.applications.vgg16.preprocess_input(x), y), 
    num_parallel_calls=AUTOTUNE
)
val_ds = val_ds.map(
    lambda x, y: (tf.keras.applications.vgg16.preprocess_input(x), y), 
    num_parallel_calls=AUTOTUNE
)

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 4000 files belonging to 8 classes.
Using 3200 files for training.
Found 4000 files belonging to 8 classes.
Using 800 files for validation.


In [4]:
# este scope me permite, con la configuración que se declaro en el primer chunk 
# "strategy = tf.distribute.MirroredStrategy()" dividir los datos en las 2 gpus, pero
# unifica el aprendizaje
with strategy.scope():
    base_model = tf.keras.applications.VGG16(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights='imagenet'
    )
    
    base_model.trainable = False
    
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = data_augmentation(inputs)
    x = base_model(x, training=False)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = tf.keras.Model(inputs, outputs)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
